# Frozen EEG-to-primitives benchmark

This is a decoder/representation benchmark. The EEG encoder is frozen: we test whether its geometry embedding $f_g$ predicts a compact set of three oriented cuboids. The primary evidence is rendered point-cloud geometry on held-out object identities, not training reconstruction or token error alone.

## 1. Hypotheses

A category template establishes a no-object-specific-EEG baseline. A linear projection tests linear accessibility of primitive geometry from $f_g$; an MLP tests whether a small nonlinear decoder improves it. No cross-attention is used because the frozen encoder exposes one pooled vector, not a token sequence.

In [ ]:
from pathlib import Path
import os, sys, numpy as np, torch
from torch.utils.data import DataLoader, Dataset
ROOT = Path.cwd().resolve()
if not (ROOT / 'primitive_decoder').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from primitive_decoder.splits import make_object_splits, verify_no_leakage
from primitive_decoder.targets import load_or_fit_targets
from primitive_decoder.embeddings import load_embedding_cache
from primitive_decoder.models import LinearCuboidDecoder, MLPCuboidDecoder
from primitive_decoder.training import fit_decoder, category_template_baseline, conditioning_controls, render_cuboids_torch
from primitive_decoder.metrics import geometry_metrics
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEEDS, STEPS, BATCH_SIZE = (11, 29, 47), 3000, 16
DATA = ROOT / 'data/neuro-3d'
TARGET_FILE = DATA / 'processed/v1/k3_cuboids_all/fitted_cuboids.npz'
FG_CACHE = Path(os.environ.get('N3D_FG_CACHE', DATA / 'processed/v1/sub01_frozen_fg.npz'))
assert (DATA / 'extracted/sub01/name_label.npy').is_file(), 'Run the documented dataset setup first.'
assert FG_CACHE.is_file(), 'Set N3D_FG_CACHE to a frozen, repeat-averaged f_g cache (labels, fg); do not retrain the encoder.'
print({'device': DEVICE, 'target_file': str(TARGET_FILE), 'fg_cache': str(FG_CACHE)})

## 2. Targets, frozen embeddings, and object-disjoint splits

Each token is `[center(3), log half-extents(3), rotation-6D(6)]`. Targets are generated by the repository's deterministic K=3 cuboid fitter. Variants 00-06 are train, 07 validation, and 08-09 held-out test. The fixed ten-object primitive benchmark is never train data.

In [ ]:
labels = np.load(DATA / 'extracted/sub01/name_label.npy', allow_pickle=False).astype(str)
split = make_object_splits(labels.tolist())
verify_no_leakage(split, ROOT / 'benchmark/primitive_10_objects.csv')
targets = load_or_fit_targets(TARGET_FILE, data_root=DATA, labels=sorted(set(labels)))
token_by_label = targets.by_label()
def category(label): return label.split('_', 1)[1].rsplit('_', 1)[0]
def records(names):
    return [(name, category(name), token_by_label[name]) for name in names]
train_records, val_records, test_records = map(records, (split.train, split.validation, split.test))
print({key: len(value) for key, value in zip(('train','validation','test'), (train_records,val_records,test_records))})
assert not set(split.train) & set(split.test)
assert not set(split.train) & {row['label'] for row in __import__('csv').DictReader(open(ROOT/'benchmark/primitive_10_objects.csv'))}

## 3. Geometry inputs and target inspection

`load_embedding_cache` only accepts a 1024-D object-level frozen $f_g$ cache. Generate it with the existing frozen encoder before this experiment; the benchmark never calls `backward()` on encoder parameters.

In [ ]:
def cloud(label):
    name = label.split('_', 1)[1]
    return np.load(DATA / 'extracted/point_cloud' / f'{name}.npy')[:, :3].astype('float32')
class CuboidData(Dataset):
    def __init__(self, rows):
        self.labels = [row[0] for row in rows]; self.categories = [row[1] for row in rows]
        self.fg = torch.from_numpy(load_embedding_cache(FG_CACHE, self.labels))
        self.tokens = torch.from_numpy(np.stack([row[2] for row in rows]))
        self.points = torch.from_numpy(np.stack([cloud(label) for label in self.labels]))
    def __len__(self): return len(self.labels)
    def __getitem__(self, index): return {'fg': self.fg[index], 'tokens': self.tokens[index], 'points': self.points[index]}
train_ds, val_ds, test_ds = map(CuboidData, (train_records, val_records, test_records))
print(train_ds.fg.shape, train_ds.tokens.shape, test_ds.points.shape)
# Visualize a target with the project explorer or render_cuboids_torch below before interpreting decoder results.

## 4. Identical-budget baselines

All learned models use the same train/validation split, three seeds, optimizer budget, and frozen embeddings. The objective combines permutation/symmetry-aware token loss with differentiable rendered Chamfer loss. The rendered loss and held-out rendered metrics are the primary quantities.

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
def train_family(factory):
    results = []
    for seed in SEEDS:
        result = fit_decoder(factory(), train_loader, val_loader, steps=STEPS, seed=seed, device=DEVICE)
        model = factory().to(DEVICE); model.load_state_dict(result.state_dict); model.eval()
        results.append((seed, model, result.history))
    return results
linear_runs = train_family(LinearCuboidDecoder)
mlp_runs = train_family(MLPCuboidDecoder)
template_tokens = category_template_baseline(train_ds.tokens.to(DEVICE), np.asarray(train_ds.categories), test_ds.categories)
print('finished:', len(linear_runs), 'linear and', len(mlp_runs), 'MLP seeds')

## 5. Held-out rendered-geometry evaluation

Metrics are symmetric squared Chamfer, cloud-diagonal-normalized Chamfer, and F-score. Each cuboid set is rendered to a point cloud before scoring. Report per-object values and aggregate mean ± standard deviation across seeds.

In [ ]:
def score_tokens(tokens, labels):
    rendered = render_cuboids_torch(tokens.to(DEVICE)).detach().cpu().numpy()
    return [{**{'label': label}, **geometry_metrics(pred, cloud(label))} for label, pred in zip(labels, rendered)]
rows = {'template': score_tokens(template_tokens, test_ds.labels)}
for name, runs in [('linear', linear_runs), ('mlp', mlp_runs)]:
    rows[name] = [score_tokens(model(test_ds.fg.to(DEVICE)), test_ds.labels) for _, model, _ in runs]
import pandas as pd
for name, value in rows.items():
    table = pd.DataFrame(value if name == 'template' else [item for seed in value for item in seed])
    print(name, table.groupby('label').mean(numeric_only=True).mean().to_dict())

## 6. Conditioning controls and interpretation

For each trained MLP, compare normal, zero, and shuffled $f_g$ on exactly the same test objects. If zeroed or shuffled embeddings match normal performance, the decoder is not demonstrating object-specific EEG conditioning. Training-set reconstruction is not evidence of EEG decoding; this benchmark does not establish causal neural representation or superiority to point-cloud diffusion.

In [ ]:
seed, best_mlp, _ = mlp_runs[0]
controls = conditioning_controls(best_mlp, test_ds.fg.to(DEVICE), seed=0)
control_rows = {name: score_tokens(value, test_ds.labels) for name, value in controls.items()}
for name, value in control_rows.items():
    print(name, pd.DataFrame(value).mean(numeric_only=True).to_dict())
# Save metrics, checkpoints, and matched-camera visualizations in a versioned results directory before making conclusions.